# Phase 3 — Coupled Thermodynamic Simulation (Dryer, Sand Storage & NCG Recovery)

**Project:** EcoFarm Circular — AI-Based Intelligent Energy Management System  
**Provisional Site:** Settat, Casablanca-Settat, Morocco (`33.0010° N`, `-7.6166° E`)

## Learning Objectives
1. Compute biomass dryer water evaporation mass balance and First-Law non-overlapping enthalpy terms coupled via Magnus-Tetens psychrometrics to real Phase 2 Settat hourly weather.
2. Simulate sensible silica sand thermal energy storage ($Q = m C_p \Delta T / 3600$) with conductive/convective standby losses ($UA \cdot \Delta T$) and hardwired PLC temperature limits.
3. Model ISO 9806 solar thermal collectors and configurable pyrolysis Non-Condensable Gas (NCG) heat recovery.
4. Export separate CSV datasets for Biomass Dryer, Sand Storage, NCG Recovery, and Integrated Thermal Balances (`Low`, `Medium`, `High`).

## Key Physical Equations
- **Wet-Basis Evaporation Rate:** $\dot{m}_{\text{evap}} = \dot{m}_{\text{wet}} \cdot \frac{X_i - X_f}{1 - X_f}$
- **Sand Bed Energy Step:** $\frac{dT_{\text{sand}}}{dt} = \frac{Q_{\text{charge}} - Q_{\text{discharge}} - UA(T_{\text{sand}} - T_{\text{amb}})}{m_{\text{sand}} C_{p,\text{sand}}}$


In [ ]:
# Google Colab & Portable Environment Bootstrap (Relative Paths + Optional Drive/Upload)
import os, sys, zipfile, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print('Using Colab built-in scientific Python stack (pandas, numpy, scikit-learn, matplotlib, seaborn).')

# Option A (Recommended in Colab): Mount Google Drive if EcoFarm_AI is saved on Drive
USE_GOOGLE_DRIVE = False
if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust folder name below if stored elsewhere in Google Drive:
    os.chdir('/content/drive/MyDrive/EcoFarm_AI')

# Resolve EcoFarm_AI project root reliably from notebooks/, root, or Colab /content/
cwd = Path.cwd().resolve()
if (cwd / 'src').is_dir():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'src').is_dir():
    PROJECT_ROOT = cwd.parent
elif (cwd / 'EcoFarm_AI' / 'src').is_dir():
    PROJECT_ROOT = cwd / 'EcoFarm_AI'
else:
    PROJECT_ROOT = cwd

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Environment   :', 'Google Colab' if IN_COLAB else 'Local / Jupyter')
print('PROJECT_ROOT  :', PROJECT_ROOT)


In [ ]:
# Dataset & Module Availability Check (with Optional Colab Upload Prompt)
required_files = [
    'src/thermal_model.py',
    'src/storage_model.py',
    'src/simulation.py',
    'data/processed/solar_weather_merged_clean.csv',
]
missing = [p for p in required_files if not (PROJECT_ROOT / p).is_file()]
print(f'Verified {len(required_files) - len(missing)}/{len(required_files)} required project files in {PROJECT_ROOT}.')

if missing:
    print('Missing relative files:', missing)
    if IN_COLAB:
        from google.colab import files
        print('Upload the missing EcoFarm CSV / Phase ZIP files below:')
        uploaded = files.upload()
        for fname in uploaded:
            if fname.endswith('.zip'):
                with zipfile.ZipFile(fname, 'r') as zf:
                    zf.extractall(PROJECT_ROOT.parent if PROJECT_ROOT.name == 'EcoFarm_AI' else PROJECT_ROOT)
                print('Extracted uploaded archive:', fname)


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from src.config import SIMULATED_DATA_DIR
from src.simulation import run_phase3_simulations

# Execute all 3 thermodynamic scenarios (Low, Medium, High) against real 2020-2023 Settat weather
report = run_phase3_simulations()
for sc_name, sc in report['scenarios'].items():
    s4 = sc['summary_4yr']
    v = sc['validation']
    print(f"[{sc_name.upper():6s}] PV={s4['pv_electrical_mwh_e']:.2f} MWh_e | SolTh={s4['solar_thermal_mwh_th']:.2f} MWh_th | NCG={s4['ncg_recovered_mwh_th']:.2f} MWh_th | Cov={s4['renewable_recovered_coverage_pct']:.1f}% | BalanceErr={v['max_energy_balance_error_kwh']:.2e} kWh")


In [ ]:
# Plot Sample Week of Coupled Sand Temperature & Thermal Flows (Medium Scenario, April 2023)
df_med = pd.read_csv(SIMULATED_DATA_DIR / 'thermal_sim_medium.csv', parse_dates=['timestamp'])
apr_mask = (df_med['timestamp'] >= '2023-04-10') & (df_med['timestamp'] <= '2023-04-17')
sub = df_med.loc[apr_mask]

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(sub['timestamp'], sub['sand_temp_c'], color='#B45309', lw=2, label='Sand Bed Temp (°C)')
axes[0].axhline(120.0, color='#DC2626', ls='--', lw=1, label='T_min Floor (120 °C)')
axes[0].set_ylabel('Temp (°C)'); axes[0].legend(); axes[0].set_title('Phase 3 Simulated Sand Bed & Thermal Balance (Apr 2023)')

axes[1].plot(sub['timestamp'], sub['solar_thermal_power_kw'], color='#D97706', label='Solar Thermal (kW_th)')
axes[1].plot(sub['timestamp'], sub['dryer_thermal_demand_kw'], color='#0F172A', ls='--', label='Dryer Demand (kW_th)')
axes[1].plot(sub['timestamp'], sub['recovered_thermal_power_kw'], color='#059669', label='NCG Recovery (kW_th)')
axes[1].set_ylabel('Thermal Power (kW_th)'); axes[1].legend()
plt.tight_layout(); plt.show()


## Conclusions & Limitations
- Energy balance residuals across `35,064` simulated hours remain within numerical tolerance (`< 1e-3 kWh`).
- Simulated curves are deterministic `simulated_physics` trajectories conditioned on assumed plant sizing and configurable operating schedules.


In [ ]:
# Export & Download Phase 3 Package (CSV, XLSX, Notebook, Code, Figures & Reports)
export_dir = PROJECT_ROOT / 'exports' / 'phase_03'
export_dir.mkdir(parents=True, exist_ok=True)
zip_path = export_dir / 'EcoFarm_Phase03_Package.zip'

patterns = ['src/config.py', 'src/thermal_model.py', 'src/storage_model.py', 'src/simulation.py', 'docs/phase3_thermal_validation.md', 'data/simulated/*.csv', 'data/simulated/*.json', 'notebooks/phase3_thermal_simulation.ipynb']
added = set()
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for pat in patterns:
        for fp in sorted(PROJECT_ROOT.glob(pat)):
            if fp.is_file() and '__pycache__' not in fp.parts and fp not in added:
                added.add(fp)
                arc = (Path('EcoFarm_AI') / fp.relative_to(PROJECT_ROOT)).as_posix()
                zi = zipfile.ZipInfo(arc, date_time=(2026, 11, 21, 12, 0, 0))
                zi.create_system = 0
                zi.compress_type = zipfile.ZIP_DEFLATED
                zi.external_attr = 0o644 << 16
                zf.writestr(zi, fp.read_bytes())

with zipfile.ZipFile(zip_path, 'r') as zcheck:
    assert zcheck.testzip() is None, 'ZIP CRC check failed!'

print(f'Packaged {len(added)} verified deliverables -> {zip_path} ({zip_path.stat().st_size:,} bytes)')
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
